# DP Ph1 power-flow initialization validation

| Status | Result |
|---|---|
| Validated | Classes marked PASS in their own section start from an SP power-flow operating point and produce a finite, bumpless trace. |
| To do | 4th- and 6th-order PCM synchronous generators show post-initialization drift; the inverter harmonic source is outside fundamental-frequency PF scope. |

Each class has its own test-and-plot cell. Failed numerical checks are reported in the output and do not stop later cases.


In [ ]:
from pathlib import Path
import os, sys, math, tempfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

_repo_hint = Path(os.environ.get("DPSIM_SOURCE_DIR", Path.cwd())).expanduser().resolve()
REPO = next(
    (
        p
        for p in [_repo_hint, *_repo_hint.parents]
        if (p / "dpsim-models").is_dir() and (p / "dpsim").is_dir()
    ),
    _repo_hint,
)
BUILD = Path(os.environ.get("DPSIM_BUILD_DIR", REPO / "build")).expanduser().resolve()
if str(BUILD) not in sys.path:
    sys.path.insert(0, str(BUILD))
if not list(BUILD.glob("dpsimpy*.so")):
    for _candidate in BUILD.glob("lib*/dpsimpy*") if BUILD.is_dir() else []:
        if _candidate.is_dir():
            sys.path.insert(0, str(_candidate.parent))
            break
import dpsimpy as dps

OUT = (
    Path(
        os.environ.get(
            "DPSIM_PF_OUTPUT_DIR", tempfile.mkdtemp(prefix="dpsim_pf_initialization_")
        )
    )
    .expanduser()
    .resolve()
)
OUT.mkdir(parents=True, exist_ok=True)
dps.Logger.set_log_dir(str(OUT))
VALIDATION_RESULTS = []


def check(condition, label, detail=None):
    passed = bool(condition)
    VALIDATION_RESULTS.append(
        {
            "Check": label,
            "Status": "PASS" if passed else "FAIL",
            "Details": "" if passed else str(detail or "condition was false"),
        }
    )
    if not passed:
        print(f"FAIL — {label}: {detail or 'condition was false'}")
    return passed


F = 50.0
FREQUENCY = F
W = 2 * np.pi * F
OMEGA = W
RMS_TO_PEAK = np.sqrt(2)
DT = 1e-5
T_END = 0.04
I3 = np.eye(3)
Z3 = np.zeros((3, 3))
RMS3_TO_PEAK1 = np.sqrt(2 / 3)
print(f"dpsimpy: {dps.__file__} | output: {OUT}")

## Shared helpers

Build the three-bus SP power flow and reuse small runners for a single component at a time.

In [ ]:
SG1_FREQ = 60.0


def run_sim(
    system,
    domain,
    name,
    fields,
    final_time=T_END,
    dt=DT,
    time_step=None,
    require_finite=True,
):
    if time_step is not None:
        dt = time_step
    folder = OUT / name
    folder.mkdir(parents=True, exist_ok=True)
    dps.Logger.set_log_dir(str(folder))
    logger = dps.Logger(name)
    for alias, attr, obj in fields:
        logger.log_attribute(alias, attr, obj)
    sim = dps.Simulation(name, dps.LogLevel.off)
    sim.set_system(system)
    sim.set_domain(domain)
    sim.set_time_step(dt)
    sim.set_final_time(final_time)
    sim.add_logger(logger)
    sim.run()
    frame = pd.read_csv(folder / f"{name}.csv", skipinitialspace=True)
    numeric = frame.select_dtypes(include=[np.number]).to_numpy()
    if require_finite:
        check(
            (np.isfinite(numeric).all()),
            "validation at line 12",
            f"{name}: non-finite samples",
        )
    return frame


log_and_run = run_sim


def configure_pf(system, name):
    folder = OUT / name
    folder.mkdir(parents=True, exist_ok=True)
    dps.Logger.set_log_dir(str(folder))
    sim = dps.Simulation(name, dps.LogLevel.off)
    sim.set_system(system)
    sim.set_domain(dps.Domain.SP)
    sim.set_time_step(0.01)
    sim.set_final_time(0.02)
    sim.set_solver(dps.Solver.NRP)
    sim.set_solver_component_behaviour(dps.SolverBehaviour.Initialization)
    sim.do_init_from_nodes_and_terminals(True)
    sim.run()


configure_pf_sim = configure_pf


def make_pf(p=10e3, q=2e3, *, name="Load", branch_g=0.0, branch_b=0.0):
    n0 = dps.sp.SimNode("bus0", dps.PhaseType.Single)
    n1 = dps.sp.SimNode("bus1", dps.PhaseType.Single)
    n2 = dps.sp.SimNode("bus2", dps.PhaseType.Single)
    src = dps.sp.ph1.NetworkInjection("Slack")
    src.set_parameters(400.0)
    src.set_base_voltage(400.0)
    src.modify_power_flow_bus_type(dps.PowerflowBusType.VD)
    src.connect([n0])
    l01 = dps.sp.ph1.PiLine("Line01")
    l01.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
    l01.set_base_voltage(400)
    l01.connect([n0, n1])
    l12 = dps.sp.ph1.PiLine("Line12")
    l12.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
    l12.set_base_voltage(400)
    l12.connect([n1, n2])
    sh = dps.sp.ph1.Shunt("Shunt")
    sh.set_parameters(branch_g, branch_b)
    sh.set_base_voltage(400)
    sh.connect([n1])
    load = dps.sp.ph1.Load(name)
    load.set_parameters(p, q, 400.0)
    load.modify_power_flow_bus_type(dps.PowerflowBusType.PQ)
    load.connect([n2])
    pf = dps.SystemTopology(F, [n0, n1, n2], [src, l01, l12, sh, load])
    configure_pf(pf, "PF_" + name)
    return pf, (n0, n1, n2)


def make_dp_ph1(
    pf,
    load_class="RXLoad",
    *,
    load_name="Load",
    load_parameters=None,
    shunt_parameters=(0.0, 0.0),
):
    n0 = dps.dp.SimNode("bus0")
    n1 = dps.dp.SimNode("bus1")
    n2 = dps.dp.SimNode("bus2")
    src = dps.dp.ph1.NetworkInjection("Slack")
    l01 = dps.dp.ph1.PiLine("Line01")
    l01.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
    l12 = dps.dp.ph1.PiLine("Line12")
    l12.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
    sh = dps.dp.ph1.Shunt("Shunt")
    if shunt_parameters is not None:
        sh.set_parameters(*shunt_parameters)
    load = getattr(dps.dp.ph1, load_class)(load_name)
    if load_parameters is not None:
        load.set_parameters(*load_parameters)
    src.connect([n0])
    l01.connect([n0, n1])
    l12.connect([n1, n2])
    sh.connect([n1])
    load.connect([n2])
    system = dps.SystemTopology(F, [n0, n1, n2], [src, l01, l12, sh, load])
    system.init_with_powerflow(pf, dps.Domain.DP)
    return system, (n0, n1, n2), src, (l01, l12), sh, load


def col(frame, name, complex_value=False):
    if complex_value:
        r = next(c for c in (name + "_0.re", name + ".re") if c in frame)
        i = next(c for c in (name + "_0.im", name + ".im") if c in frame)
        return frame[r].to_numpy() + 1j * frame[i].to_numpy()
    k = next(c for c in (name + "_0", name) if c in frame)
    return frame[k].to_numpy()


signal_column = col


def run_dp_ph1_primitive(model, p, q, value, label):
    pf, pfn = make_pf(p, q, name="PFAnchor")
    n0 = dps.dp.SimNode("bus0")
    n1 = dps.dp.SimNode("bus1")
    n2 = dps.dp.SimNode("bus2")
    source = dps.dp.ph1.NetworkInjection("Slack")
    a = dps.dp.ph1.PiLine("Line01")
    a.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
    b = dps.dp.ph1.PiLine("Line12")
    b.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
    comp = getattr(dps.dp.ph1, model)("Test_" + label)
    if isinstance(value, tuple):
        comp.set_parameters(*value)
    else:
        comp.set_parameters(value)
    source.connect([n0])
    a.connect([n0, n1])
    b.connect([n1, n2])
    comp.connect([n2, dps.dp.SimNode.gnd])
    system = dps.SystemTopology(F, [n0, n1, n2], [source, a, b, comp])
    system.init_with_powerflow(pf, dps.Domain.DP)
    frame = run_sim(
        system,
        dps.Domain.DP,
        "DP_Ph1_Primitive_" + label,
        [("v", "v", n2), ("i", "i_intf", comp)],
        final_time=0.002,
    )
    v = col(frame, "v", True)
    i = col(frame, "i", True)
    check(
        (abs(v[0] - pfn[2].single_voltage()) / abs(pfn[2].single_voltage()) < 1e-6),
        "validation at line 72",
        "validation condition is false",
    )
    check(
        (np.isfinite(i).all()), "validation at line 73", "validation condition is false"
    )
    relative_ripple = float(np.max(np.abs(i - i[0])) / max(abs(i[0]), 1e-9))
    check((relative_ripple < 2e-2), "validation at line 75", (label, relative_ripple))
    return {
        "Model": label,
        "PF start voltage [V]": abs(v[0]),
        "initial current [A]": abs(i[0]),
        "first 2ms envelope ripple [%]": 100 * relative_ripple,
        "trace": frame,
    }


def run_dp_ph3_primitive(model, p, q, value, label):
    pf, pfn = make_pf(p, q, name="PFAnchor")
    n0 = dps.dp.SimNode("bus0", dps.PhaseType.ABC)
    n1 = dps.dp.SimNode("bus1", dps.PhaseType.ABC)
    n2 = dps.dp.SimNode("bus2", dps.PhaseType.ABC)
    source = dps.dp.ph3.NetworkInjection("Slack")
    a = dps.dp.ph3.PiLine("Line01")
    a.set_parameters(0.01 * I3, 0.1e-3 * I3, 1e-6 * I3, 1e-6 * I3)
    b = dps.dp.ph3.PiLine("Line12")
    b.set_parameters(0.02 * I3, 0.2e-3 * I3, 1e-6 * I3, 1e-6 * I3)
    comp = getattr(dps.dp.ph3, model)("Test_" + label)
    comp.set_parameters(value * I3)
    source.connect([n0])
    a.connect([n0, n1])
    b.connect([n1, n2])
    comp.connect([n2, dps.dp.SimNode.gnd])
    system = dps.SystemTopology(F, [n0, n1, n2], [source, a, b, comp])
    system.init_with_powerflow(pf, dps.Domain.DP)
    frame = run_sim(
        system,
        dps.Domain.DP,
        "DP_Ph3_Primitive_" + label,
        [("v", "v", n2), ("i", "i_intf", comp)],
        final_time=0.002,
    )
    v = col(frame, "v", True)
    i = col(frame, "i", True)
    check(
        (np.isfinite(v).all() and np.isfinite(i).all()), "validation at line 89", label
    )
    ripple = float(np.max(np.abs(i - i[0])) / max(abs(i[0]), 1e-9))
    check((ripple < 2e-2), "validation at line 91", (label, ripple))
    return {
        "Model": label,
        "phase-A initial current [A peak]": abs(i[0]),
        "first 2ms envelope ripple [%]": 100 * ripple,
        "trace": frame,
    }


def run_ph1_closed_switch(cls, label, variable=False):
    pf, pfn = make_pf(10e3, 0.0, name="PFAnchor")
    n0 = dps.dp.SimNode("bus0")
    n1 = dps.dp.SimNode("bus1")
    n2 = dps.dp.SimNode("bus2")
    src = dps.dp.ph1.NetworkInjection("Slack")
    a = dps.dp.ph1.PiLine("Line01")
    a.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
    b = dps.dp.ph1.PiLine("Line12")
    b.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
    sw = getattr(dps.dp.ph1, cls)("Test_" + label)
    sw.set_parameters(1e9, 16.0, True)
    if variable:
        sw.set_init_parameters(DT)
    src.connect([n0])
    a.connect([n0, n1])
    b.connect([n1, n2])
    sw.connect([n2, dps.dp.SimNode.gnd])
    system = dps.SystemTopology(F, [n0, n1, n2], [src, a, b, sw])
    system.init_with_powerflow(pf, dps.Domain.DP)
    frame = run_sim(
        system,
        dps.Domain.DP,
        "DP_Ph1_" + label,
        [("v", "v", n2), ("i", "i_intf", sw)],
        final_time=0.002,
    )
    v = col(frame, "v", True)
    i = col(frame, "i", True)
    check(
        (np.isfinite(v).all() and np.isfinite(i).all()),
        "validation at line 106",
        "validation condition is false",
    )
    ripple = float(np.max(np.abs(i - i[0])) / max(abs(i[0]), 1e-9))
    check((ripple < 2e-2), "validation at line 107", (label, ripple))
    return {
        "Domain": "DP Ph1",
        "Model": label,
        "first 2ms current ripple [%]": 100 * ripple,
        "trace": frame,
    }


def run_ph3_series(model, label):
    pf, pfn = make_pf(10e3, 0.0, name="PFAnchor")
    n0 = dps.dp.SimNode("bus0", dps.PhaseType.ABC)
    n1 = dps.dp.SimNode("bus1", dps.PhaseType.ABC)
    n2 = dps.dp.SimNode("bus2", dps.PhaseType.ABC)
    src = dps.dp.ph3.NetworkInjection("Slack")
    a = dps.dp.ph3.PiLine("Line01")
    a.set_parameters(0.01 * I3, 0.1e-3 * I3, 1e-6 * I3, 1e-6 * I3)
    b = dps.dp.ph3.PiLine("Line12")
    b.set_parameters(0.02 * I3, 0.2e-3 * I3, 1e-6 * I3, 1e-6 * I3)
    comp = getattr(dps.dp.ph3, model)("Test_" + label)
    if model == "SeriesResistor":
        comp.set_parameters(16.0)
    elif model == "SeriesSwitch":
        comp.set_parameters(1e9, 16.0, True)
    else:
        comp.set_parameters(1e9 * I3, 16.0 * I3, True)
    src.connect([n0])
    a.connect([n0, n1])
    b.connect([n1, n2])
    comp.connect([n2, dps.dp.SimNode.gnd])
    system = dps.SystemTopology(F, [n0, n1, n2], [src, a, b, comp])
    system.init_with_powerflow(pf, dps.Domain.DP)
    frame = run_sim(
        system,
        dps.Domain.DP,
        "DP_Ph3_" + label,
        [("v", "v", n2), ("i", "i_intf", comp)],
        final_time=0.002,
    )
    v = col(frame, "v", True)
    i = col(frame, "i", True)
    check(
        (np.isfinite(v).all() and np.isfinite(i).all()),
        "validation at line 123",
        "validation condition is false",
    )
    ripple = float(np.max(np.abs(i - i[0])) / max(abs(i[0]), 1e-9))
    check((ripple < 2e-2), "validation at line 124", (label, ripple))
    return {
        "Domain": "DP Ph3",
        "Model": label,
        "first 2ms current ripple [%]": 100 * ripple,
        "trace": frame,
    }


# Sweep every DP Ph1 reduced-order synchronous-generator implementation exposed by pybind.
# Each machine receives parameters only; operating-point states must come through init_with_powerflow.
SYNC_PU = dict(
    nom_power=100e6,
    nom_voltage=24e3,
    nom_frequency=SG1_FREQ,
    H=3.5,
    Ld=1.8,
    Lq=1.7,
    L0=0.1,
    Ld_t=0.3,
    Lq_t=0.25,
    Td0_t=8.0,
    Tq0_t=0.4,
    Ld_s=0.2,
    Lq_s=0.2,
    Td0_s=0.03,
    Tq0_s=0.05,
    Taa=0.0,
)


def run_reduced_generator_case(model):
    n0 = dps.dp.SimNode("SG_Gen")
    n1 = dps.dp.SimNode("SG_Slack")
    gen = getattr(dps.dp.ph1, model)("SG_Gen")
    if model == "SynchronGenerator3OrderVBR":
        gen.set_operational_parameters_per_unit(
            SYNC_PU["nom_power"],
            SYNC_PU["nom_voltage"],
            SG1_FREQ,
            SYNC_PU["H"],
            SYNC_PU["Ld"],
            SYNC_PU["Lq"],
            SYNC_PU["L0"],
            SYNC_PU["Ld_t"],
            SYNC_PU["Td0_t"],
        )
    elif model in (
        "SynchronGenerator4OrderVBR",
        "SynchronGenerator4OrderTPM",
        "SynchronGenerator4OrderPCM",
    ):
        gen.set_operational_parameters_per_unit(
            SYNC_PU["nom_power"],
            SYNC_PU["nom_voltage"],
            SG1_FREQ,
            SYNC_PU["H"],
            SYNC_PU["Ld"],
            SYNC_PU["Lq"],
            SYNC_PU["L0"],
            SYNC_PU["Ld_t"],
            SYNC_PU["Lq_t"],
            SYNC_PU["Td0_t"],
            SYNC_PU["Tq0_t"],
        )
    else:
        gen.set_operational_parameters_per_unit(
            SYNC_PU["nom_power"],
            SYNC_PU["nom_voltage"],
            SG1_FREQ,
            SYNC_PU["H"],
            SYNC_PU["Ld"],
            SYNC_PU["Lq"],
            SYNC_PU["L0"],
            SYNC_PU["Ld_t"],
            SYNC_PU["Lq_t"],
            SYNC_PU["Td0_t"],
            SYNC_PU["Tq0_t"],
            SYNC_PU["Ld_s"],
            SYNC_PU["Lq_s"],
            SYNC_PU["Td0_s"],
            SYNC_PU["Tq0_s"],
            SYNC_PU["Taa"],
        )
    line = dps.dp.ph1.PiLine("SG_Line")
    line.set_parameters(0.1, 1e-3, 1e-12, 1e-6)
    slack = dps.dp.ph1.NetworkInjection("SG_Slack")
    gen.connect([n0])
    line.connect([n0, n1])
    slack.connect([n1])
    system = dps.SystemTopology(SG1_FREQ, [n0, n1], [gen, line, slack])
    system.init_with_powerflow(sg_pf_sys, dps.Domain.DP)
    name = "DP_Ph1_" + model + "_AutoPF"
    frame = run_sim(
        system,
        dps.Domain.DP,
        name,
        [("v", "v", n0), ("i", "i_intf", gen)],
        final_time=1e-3,
        require_finite=False,
    )
    v = col(frame, "v", True)
    i = col(frame, "i", True)
    check(
        (np.isfinite(v[0]) and np.isfinite(i[0])),
        "validation at line 153",
        (model, "non-finite initial operating point"),
    )
    check(
        (abs(v[0] - sg0.single_voltage()) / abs(sg0.single_voltage()) < 1e-6),
        "validation at line 154",
        (model, "PF voltage mismatch", v[0]),
    )
    finite = bool(np.isfinite(v).all() and np.isfinite(i).all())
    drift = float(np.nanmax(np.abs(i - i[0])) / max(abs(i[0]), 1e-9) * 100)
    status = "BUMPLESS" if finite and drift < 2.0 else "FAIL: unstable/drifting"
    return {
        "Domain/class": "DP.Ph1." + model,
        "PF voltage error [%]": 100
        * abs(v[0] - sg0.single_voltage())
        / abs(sg0.single_voltage()),
        "1 ms current drift [%]": drift,
        "finite trace": finite,
        "result": status,
        "trace": frame,
    }


def run_pf_mapped_voltage_source(phase, source_class):
    pf, pfn = make_pf(10e3, 2e3, name="Load")
    n0 = (
        dps.dp.SimNode("bus0", dps.PhaseType.ABC)
        if phase == "Ph3"
        else dps.dp.SimNode("bus0")
    )
    n1 = (
        dps.dp.SimNode("bus1", dps.PhaseType.ABC)
        if phase == "Ph3"
        else dps.dp.SimNode("bus1")
    )
    n2 = (
        dps.dp.SimNode("bus2", dps.PhaseType.ABC)
        if phase == "Ph3"
        else dps.dp.SimNode("bus2")
    )
    mod = getattr(dps.dp, phase.lower())
    src = getattr(mod, source_class)("Slack")
    if source_class == "VoltageSourceNorton":
        src.set_parameters(0j, -1.0, 0.1)
    if phase == "Ph3":
        a = mod.PiLine("Line01")
        a.set_parameters(0.01 * I3, 0.1e-3 * I3, 1e-6 * I3, 1e-6 * I3)
        b = mod.PiLine("Line12")
        b.set_parameters(0.02 * I3, 0.2e-3 * I3, 1e-6 * I3, 1e-6 * I3)
        load = mod.CurrentSource("Load")
        src.connect([dps.dp.SimNode.gnd, n0])
        a.connect([n0, n1])
        b.connect([n1, n2])
        load.connect([dps.dp.SimNode.gnd, n2])
        comps = [src, a, b, load]
    else:
        a = mod.PiLine("Line01")
        a.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
        b = mod.PiLine("Line12")
        b.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
        load = mod.RXLoad("Load")
        src.connect([dps.dp.SimNode.gnd, n0])
        a.connect([n0, n1])
        b.connect([n1, n2])
        load.connect([n2])
        comps = [src, a, b, load]
    system = dps.SystemTopology(F, [n0, n1, n2], comps)
    system.init_with_powerflow(pf, dps.Domain.DP)
    frame = run_sim(
        system,
        dps.Domain.DP,
        "DP_" + phase + "_" + source_class + "_AutoPF",
        [("v", "v", n0)],
        final_time=0.002,
    )
    actual = col(frame, "v", True)[0]
    expected = (
        pfn[0].single_voltage()
        if phase == "Ph1"
        else RMS3_TO_PEAK1 * pfn[0].single_voltage()
    )
    error = float(abs(actual - expected) / max(abs(expected), 1e-12))
    check(
        (np.isfinite(actual) and error < 1e-6),
        "validation at line 185",
        (phase, source_class, "PF voltage mismatch", actual, expected, error),
    )
    return {
        "Domain/class": "DP." + phase + "." + source_class,
        "PF voltage error [%]": 100 * error,
        "finite trace": True,
        "trace": frame,
    }


def run_generic_v_ssn(phase):
    pf, pfn = make_pf(10e3, 2e3, name="Load")
    if phase == "Ph1":
        n0 = dps.dp.SimNode("bus0")
        n1 = dps.dp.SimNode("bus1")
        n2 = dps.dp.SimNode("bus2")
        mod = dps.dp.ph1
        ident = np.ones((1, 1))
        scale = 1.0
        a = mod.PiLine("Line01")
        a.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
        b = mod.PiLine("Line12")
        b.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
    else:
        n0 = dps.dp.SimNode("bus0", dps.PhaseType.ABC)
        n1 = dps.dp.SimNode("bus1", dps.PhaseType.ABC)
        n2 = dps.dp.SimNode("bus2", dps.PhaseType.ABC)
        mod = dps.dp.ph3
        ident = I3
        scale = 1.0
        a = mod.PiLine("Line01")
        a.set_parameters(0.01 * I3, 0.1e-3 * I3, 1e-6 * I3, 1e-6 * I3)
        b = mod.PiLine("Line12")
        b.set_parameters(0.02 * I3, 0.2e-3 * I3, 1e-6 * I3, 1e-6 * I3)
    vpf = pfn[2].single_voltage()
    power = complex(10e3, 2e3)
    z = abs(vpf) ** 2 / np.conj(power)
    r = float(z.real)
    ell = float(z.imag / W)
    comp = mod.GenericTwoTerminalVTypeSSN("Load")
    A = np.array([[-r / ell]]) * ident
    B = np.array([[1 / ell]]) * ident
    C = ident
    D = np.zeros_like(ident)
    comp.set_parameters(A, B, C, D)
    src = mod.NetworkInjection("Slack")
    if phase == "Ph1":
        src.connect([n0])
        a.connect([n0, n1])
        b.connect([n1, n2])
        comp.connect([dps.dp.SimNode.gnd, n2])
    else:
        src.connect([n0])
        a.connect([n0, n1])
        b.connect([n1, n2])
        comp.connect([dps.dp.SimNode.gnd, n2])
    system = dps.SystemTopology(F, [n0, n1, n2], [src, a, b, comp])
    system.init_with_powerflow(pf, dps.Domain.DP)
    frame = run_sim(
        system,
        dps.Domain.DP,
        "DP_" + phase + "_GenericTwoTerminalVTypeSSN_AutoPF",
        [("v", "v_intf", comp), ("i", "i_intf", comp), ("x", "x", comp)],
        final_time=0.002,
        require_finite=False,
    )
    v = col(frame, "v", True)
    i = col(frame, "i", True)
    initial = float(
        abs(v[0] - (vpf if phase == "Ph1" else RMS3_TO_PEAK1 * vpf))
        / max(abs(v[0]), 1e-12)
    )
    finite = bool(
        np.isfinite(frame.select_dtypes(include=[np.number]).to_numpy()).all()
    )
    ripple = float(np.nanmax(abs(i - i[0])) / max(abs(i[0]), 1e-9) * 100)
    check(
        (np.isfinite(v[0]) and initial < 1e-5),
        "validation at line 216",
        (phase, "V-type SSN initial voltage", initial),
    )
    return {
        "Domain/class": "DP." + phase + ".GenericTwoTerminalVTypeSSN",
        "PF port voltage error [%]": 100 * initial,
        "2 ms current drift [%]": ripple,
        "finite trace": finite,
        "result": "BUMPLESS" if finite and ripple < 2 else "CHECK",
        "trace": frame,
    }


def run_generic_i_ssn(phase):
    q = -10e3
    pf, pfn = make_pf(0.0, q, name="Load")
    capacitance = -q / (W * abs(pfn[2].single_voltage()) ** 2)
    if phase == "Ph1":
        n0 = dps.dp.SimNode("bus0")
        n1 = dps.dp.SimNode("bus1")
        n2 = dps.dp.SimNode("bus2")
        mod = dps.dp.ph1
        ident = np.ones((1, 1))
        a = mod.PiLine("Line01")
        a.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
        b = mod.PiLine("Line12")
        b.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
    else:
        n0 = dps.dp.SimNode("bus0", dps.PhaseType.ABC)
        n1 = dps.dp.SimNode("bus1", dps.PhaseType.ABC)
        n2 = dps.dp.SimNode("bus2", dps.PhaseType.ABC)
        mod = dps.dp.ph3
        ident = I3
        a = mod.PiLine("Line01")
        a.set_parameters(0.01 * I3, 0.1e-3 * I3, 1e-6 * I3, 1e-6 * I3)
        b = mod.PiLine("Line12")
        b.set_parameters(0.02 * I3, 0.2e-3 * I3, 1e-6 * I3, 1e-6 * I3)
    comp = mod.GenericTwoTerminalITypeSSN("Load")
    comp.set_parameters(
        np.zeros_like(ident), ident / capacitance, ident, np.zeros_like(ident)
    )
    src = mod.NetworkInjection("Slack")
    src.connect([n0])
    a.connect([n0, n1])
    b.connect([n1, n2])
    comp.connect([dps.dp.SimNode.gnd, n2])
    system = dps.SystemTopology(F, [n0, n1, n2], [src, a, b, comp])
    system.init_with_powerflow(pf, dps.Domain.DP)
    frame = run_sim(
        system,
        dps.Domain.DP,
        "DP_" + phase + "_GenericTwoTerminalITypeSSN_AutoPF",
        [("v", "v_intf", comp), ("i", "i_intf", comp), ("x", "x", comp)],
        final_time=0.002,
        require_finite=False,
    )
    v = col(frame, "v", True)
    i = col(frame, "i", True)
    v_expected = (
        pfn[2].single_voltage()
        if phase == "Ph1"
        else RMS3_TO_PEAK1 * pfn[2].single_voltage()
    )
    initial = float(abs(v[0] - v_expected) / max(abs(v_expected), 1e-12))
    finite = bool(
        np.isfinite(frame.select_dtypes(include=[np.number]).to_numpy()).all()
    )
    ripple = float(
        np.nanmax(np.abs(np.abs(i) - abs(i[0]))) / max(abs(i[0]), 1e-9) * 100
    )
    check(
        (np.isfinite(v[0]) and initial < 1e-5),
        "validation at line 244",
        (phase, "I-type SSN initial voltage", initial, v[0], v_expected),
    )
    check(
        (np.isfinite(i[0])),
        "validation at line 245",
        (phase, "I-type SSN initial current", i[0]),
    )
    return {
        "Domain/class": "DP." + phase + ".GenericTwoTerminalITypeSSN",
        "PF port voltage error [%]": 100 * initial,
        "2 ms current drift [%]": ripple,
        "finite trace": finite,
        "result": "BUMPLESS" if finite and ripple < 2 else "CHECK",
        "trace": frame,
    }


def run_dp_ph3_full_rlc():
    pf, pfn = make_pf(2e3, 1e3, name="Load")
    vpf = pfn[2].single_voltage()
    power = complex(2e3, 1e3)
    z = abs(vpf) ** 2 / np.conj(power)
    resistance = float(z.real)
    capacitance = 10e-6
    inductance = float((z.imag + 1 / (W * capacitance)) / W)
    n0 = dps.dp.SimNode("bus0", dps.PhaseType.ABC)
    n1 = dps.dp.SimNode("bus1", dps.PhaseType.ABC)
    n2 = dps.dp.SimNode("bus2", dps.PhaseType.ABC)
    src = dps.dp.ph3.NetworkInjection("Slack")
    a = dps.dp.ph3.PiLine("Line01")
    a.set_parameters(0.01 * I3, 0.1e-3 * I3, 1e-6 * I3, 1e-6 * I3)
    b = dps.dp.ph3.PiLine("Line12")
    b.set_parameters(0.02 * I3, 0.2e-3 * I3, 1e-6 * I3, 1e-6 * I3)
    rlc = dps.dp.ph3.Full_Serial_RLC("Load")
    rlc.set_parameters(resistance * I3, inductance * I3, capacitance * I3)
    src.connect([n0])
    a.connect([n0, n1])
    b.connect([n1, n2])
    rlc.connect([dps.dp.SimNode.gnd, n2])
    system = dps.SystemTopology(F, [n0, n1, n2], [src, a, b, rlc])
    system.init_with_powerflow(pf, dps.Domain.DP)
    frame = run_sim(
        system,
        dps.Domain.DP,
        "DP_Ph3_Full_Serial_RLC_AutoPF",
        [("v", "v_intf", rlc), ("i", "i_intf", rlc)],
        final_time=0.002,
        require_finite=False,
    )
    v = col(frame, "v", True)
    i = col(frame, "i", True)
    expected = RMS3_TO_PEAK1 * vpf
    error = float(abs(v[0] - expected) / max(abs(expected), 1e-12))
    finite = bool(
        np.isfinite(frame.select_dtypes(include=[np.number]).to_numpy()).all()
    )
    ripple = float(
        np.nanmax(np.abs(np.abs(i) - abs(i[0]))) / max(abs(i[0]), 1e-9) * 100
    )
    check(
        (np.isfinite(v[0]) and error < 2e-2),
        "validation at line 266",
        ("DP Ph3 Full_Serial_RLC initial voltage", error, v[0], expected),
    )
    return {
        "Domain/class": "DP.Ph3.Full_Serial_RLC",
        "PF port voltage error [%]": 100 * error,
        "2 ms current drift [%]": ripple,
        "finite trace": finite,
        "result": "BUMPLESS" if finite and ripple < 2 else "CHECK",
        "trace": frame,
    }


def plot_trace(frame, stem, title, complex_value=True):
    values = col(frame, stem, complex_value)
    fig, ax = plt.subplots(figsize=(9, 3.5))
    ax.plot(1e3 * frame.time, np.abs(values), label=stem)
    ax.set(title=title, xlabel="Time [ms]", ylabel=f"|{stem}| [A or V]")
    ax.grid(alpha=0.3)
    ax.legend()
    plt.show()

## CurrentSource

In [ ]:
pf_cs, pf_cs_nodes = make_pf(10e3, 2e3, name="Load")
c0 = dps.dp.SimNode("bus0")
c1 = dps.dp.SimNode("bus1")
c2 = dps.dp.SimNode("bus2")
cs_slack = dps.dp.ph1.NetworkInjection("Slack")
cs_l01 = dps.dp.ph1.PiLine("Line01")
cs_l01.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
cs_l12 = dps.dp.ph1.PiLine("Line12")
cs_l12.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
cs_load = dps.dp.ph1.CurrentSource("Load")  # Intentionally no set_parameters()
cs_slack.connect([c0])
cs_l01.connect([c0, c1])
cs_l12.connect([c1, c2])
cs_load.connect([c2, dps.dp.SimNode.gnd])
cs_sys = dps.SystemTopology(F, [c0, c1, c2], [cs_slack, cs_l01, cs_l12, cs_load])
cs_sys.init_with_powerflow(pf_cs, dps.Domain.DP)
cs_trace = run_sim(
    cs_sys,
    dps.Domain.DP,
    "DP_Ph1_CurrentSource_AutoPF",
    [("v", "v", c2), ("i", "i_intf", cs_load)],
    final_time=0.002,
)
cs_v = col(cs_trace, "v", True)
cs_i = col(cs_trace, "i", True)
cs_expected = np.conj(complex(10e3, 2e3) / pf_cs_nodes[2].single_voltage())
cs_error = abs(cs_i[0] - cs_expected) / abs(cs_expected)
cs_ripple = float(np.max(np.abs(cs_i - cs_i[0])) / max(abs(cs_i[0]), 1e-9))
check(
    (cs_error < 1e-5),
    "validation at line 14",
    ("DP Ph1 CurrentSource PF reference mismatch", cs_error, cs_i[0], cs_expected),
)
check(
    (cs_ripple < 1e-2),
    "validation at line 15",
    ("DP Ph1 CurrentSource startup bump", cs_ripple),
)
check(
    (
        abs(cs_v[0] - pf_cs_nodes[2].single_voltage())
        / abs(pf_cs_nodes[2].single_voltage())
        < 1e-6
    ),
    "validation at line 16",
    "validation condition is false",
)
print(
    f"PASS DP Ph1 CurrentSource auto-PF: current error={100*cs_error:.4g}%, first 2ms current ripple={100*cs_ripple:.4g}%"
)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(1e3 * cs_trace.time, np.abs(cs_i), label="DP Ph1 current source")
ax.set(
    title="DP Ph1 current-source load automatic PF startup",
    xlabel="Time [ms]",
    ylabel="|I| [A]",
)
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## SynchronGeneratorTrStab

In [ ]:
SG1_FREQ = 60.0
SG1_SN = 555e6
SG1_VN = 24e3
SG1_P = 50e6
sg0 = dps.sp.SimNode("SG_Gen", dps.PhaseType.Single)
sg1 = dps.sp.SimNode("SG_Slack", dps.PhaseType.Single)
sg_pf_gen = dps.sp.ph1.SynchronGenerator("SG_Gen")
sg_pf_gen.set_parameters(
    rated_apparent_power=SG1_SN,
    rated_voltage=SG1_VN,
    set_point_active_power=SG1_P,
    set_point_voltage=SG1_VN,
    powerflow_bus_type=dps.PowerflowBusType.PV,
)
sg_pf_gen.set_base_voltage(SG1_VN)
sg_pf_gen.modify_power_flow_bus_type(dps.PowerflowBusType.PV)
sg_pf_line = dps.sp.ph1.PiLine("SG_Line")
sg_pf_line.set_parameters(0.1, 1e-3, 1e-12, 1e-6)
sg_pf_line.set_base_voltage(SG1_VN)
sg_pf_slack = dps.sp.ph1.NetworkInjection("SG_Slack")
sg_pf_slack.set_parameters(SG1_VN)
sg_pf_slack.set_base_voltage(SG1_VN)
sg_pf_slack.modify_power_flow_bus_type(dps.PowerflowBusType.VD)
sg_pf_gen.connect([sg0])
sg_pf_line.connect([sg0, sg1])
sg_pf_slack.connect([sg1])
sg_pf_sys = dps.SystemTopology(
    SG1_FREQ, [sg0, sg1], [sg_pf_gen, sg_pf_line, sg_pf_slack]
)
configure_pf(sg_pf_sys, "DP_Ph1_SG_PF")
sgd0 = dps.dp.SimNode("SG_Gen")
sgd1 = dps.dp.SimNode("SG_Slack")
sg_dp = dps.dp.ph1.SynchronGeneratorTrStab("SG_Gen")
sg_dp.set_standard_parameters_PU(SG1_SN, SG1_VN, SG1_FREQ, 0.31, 3.7, 0.0, 1.5)
sg_dp_line = dps.dp.ph1.PiLine("SG_Line")
sg_dp_line.set_parameters(0.1, 1e-3, 1e-12, 1e-6)
sg_dp_slack = dps.dp.ph1.NetworkInjection("SG_Slack")
sg_dp_slack.set_parameters(SG1_VN)
sg_dp.connect([sgd0])
sg_dp_line.connect([sgd0, sgd1])
sg_dp_slack.connect([sgd1])
sg_dp_sys = dps.SystemTopology(SG1_FREQ, [sgd0, sgd1], [sg_dp, sg_dp_line, sg_dp_slack])
sg_dp_sys.init_with_powerflow(sg_pf_sys, dps.Domain.DP)
sg_trace = run_sim(
    sg_dp_sys,
    dps.Domain.DP,
    "DP_Ph1_SynchronGeneratorTrStab_AutoPF",
    [
        ("v", "v", sgd0),
        ("i", "i_intf", sg_dp),
        ("ep", "Ep", sg_dp),
        ("pe", "P_elec", sg_dp),
        ("pm", "P_mech", sg_dp),
        ("wr", "w_r", sg_dp),
    ],
    final_time=3e-4,
)
sgv = col(sg_trace, "v", True)
sgi = col(sg_trace, "i", True)
sgpe = col(sg_trace, "pe")
sgpm = col(sg_trace, "pm")
sgwr = col(sg_trace, "wr")
check(
    (np.isfinite(sg_trace.select_dtypes(include=[np.number]).to_numpy()).all()),
    "validation at line 19",
    "validation condition is false",
)
check(
    (abs(sgv[0] - sg0.single_voltage()) < 1e-6 * abs(sg0.single_voltage())),
    "validation at line 20",
    "validation condition is false",
)
check(
    (abs(sgpe[0] - sgpm[0]) < 1e-8 * max(abs(sgpe[0]), 1.0)),
    "validation at line 21",
    (sgpe[0], sgpm[0]),
)
check(
    (abs(sgwr[0] - 2 * np.pi * SG1_FREQ) < 1e-8 * 2 * np.pi * SG1_FREQ),
    "validation at line 22",
    "validation condition is false",
)
print(
    f"PASS DP Ph1 SynchronGeneratorTrStab automatic PF start: P_e=P_m={sgpe[0]:.6g} W; initial V={sgv[0]:.6g} V"
)
fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axs[0].plot(1e3 * sg_trace.time, sgi.real, label="generator current real envelope")
axs[0].set_ylabel("I [A]")
axs[0].grid(alpha=0.3)
axs[0].legend()
axs[1].plot(1e3 * sg_trace.time, sgpe, label="P_elec")
axs[1].plot(1e3 * sg_trace.time, sgpm, "--", label="P_mech")
axs[1].set(xlabel="Time [ms]", ylabel="Power [W]")
axs[1].grid(alpha=0.3)
axs[1].legend()
fig.suptitle("DP Ph1 generator PF start")
fig.tight_layout()
plt.show()

## RXLoad

In [ ]:
# DP Ph1: three-bus power flow copied automatically into DP. The RXLoad receives
# P, Q and nominal voltage from its matching PF component; no Python set_parameters call.
pf, (pf0, pf1, pf2) = make_pf(10e3, 2e3)
sys, nodes, src, lines, sh, load = make_dp_ph1(
    pf, load_class="RXLoad", load_parameters=None
)
rx = run_sim(
    sys,
    dps.Domain.DP,
    "DP_Ph1_RXLoad_AutoPF",
    [("v", "v", nodes[2]), ("i", "i_intf", load)],
    final_time=0.002,
)
rx_v = col(rx, "v", True)
rx_i = col(rx, "i", True)
expected_v = pf2.single_voltage()
expected_s = complex(10e3, 2e3)
expected_i = np.conj(expected_s / expected_v)
rx_error = abs(rx_i[0] - expected_i) / abs(expected_i)
check(
    (rx_error < 1e-6),
    "validation at line 10",
    {"RXLoad PF current mismatch": rx_error, "expected": expected_i, "actual": rx_i[0]},
)
check(
    (abs(rx_v[0] - expected_v) < 1e-6 * abs(expected_v)),
    "validation at line 11",
    "validation condition is false",
)
print(
    f"PASS DP Ph1 RXLoad automatic PF initialization: current error={100*rx_error:.3g}%"
)
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(1e3 * rx.time, rx_i.real, label="DP Ph1 load current, real envelope")
ax.set(xlabel="Time [ms]", ylabel="Current [A]")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Shunt

In [ ]:
# A second three-bus run checks passive branch and line histories against their
# PF-derived bus voltages and records the first-cycle envelope deviation.
# Shunt parameters are explicitly model parameters; its operating voltage/current
# state is initialized from the SP power flow as part of normal simulation init.
sh_pf, _ = make_pf(10e3, 0.0, name="ShuntLoad", branch_g=0.02, branch_b=0.0)
sh_sys, sh_nodes, sh_src, sh_lines, sh_dp, sh_load = make_dp_ph1(
    sh_pf,
    load_class="RXLoad",
    load_name="ShuntLoad",
    load_parameters=None,
    shunt_parameters=(0.02, 0.0),
)
sh_trace = run_sim(
    sh_sys,
    dps.Domain.DP,
    "DP_Ph1_Shunt_ThreeBus",
    [("v", "v", sh_nodes[1]), ("i", "i_intf", sh_dp)],
    final_time=0.002,
)
sh_v = col(sh_trace, "v", True)
sh_i = col(sh_trace, "i", True)
check(
    (
        abs(sh_v[0] - sh_pf.get_node("bus1").single_voltage()) < 1e-7
        if hasattr(sh_pf, "get_node")
        else np.isfinite(sh_v[0])
    ),
    "validation at line 9",
    "validation condition is false",
)
check(
    (np.isfinite(sh_i).all()), "validation at line 10", "validation condition is false"
)
print("PASS DP Ph1 Shunt and two PiLine sections start finite from a three-bus PF.")
fig, ax = plt.subplots(figsize=(10, 3))
ax.plot(1e3 * sh_trace.time, np.abs(sh_i), label="Shunt current envelope")
ax.set(xlabel="Time [ms]", ylabel="|I| [A]")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## Resistor

In [ ]:
Vnom = 400.0
result = run_dp_ph1_primitive("Resistor", 10e3, 0.0, 16.0, "Resistor")
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 Resistor PF startup")

## Inductor

In [ ]:
Vnom = 400.0
result = run_dp_ph1_primitive("Inductor", 0.0, 10e3, Vnom**2 / (W * 10e3), "Inductor")
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 Inductor PF startup")

## Capacitor

In [ ]:
Vnom = 400.0
result = run_dp_ph1_primitive(
    "Capacitor", 0.0, -10e3, 10e3 / (W * Vnom**2), "Capacitor"
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 Capacitor PF startup")

## Full_Serial_RLC

In [ ]:
Vnom = 400.0
_s = complex(2e3, 1e3)
_z = Vnom**2 / np.conj(_s)
_c = 10e-6
_l = (_z.imag + 1 / (W * _c)) / W
result = run_dp_ph1_primitive(
    "Full_Serial_RLC", 2e3, 1e3, (float(_z.real), float(_l), _c), "Full_Serial_RLC"
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 Full_Serial_RLC PF startup")

## Switch

In [ ]:
result = run_ph1_closed_switch("Switch", "Switch", False)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 Switch PF startup")

## varResSwitch

In [ ]:
result = run_ph1_closed_switch("varResSwitch", "varResSwitch", True)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 varResSwitch PF startup")

## GenericTwoTerminalVTypeSSN

In [ ]:
result = run_generic_v_ssn("Ph1")
display(pd.DataFrame([result]))
plot_trace(result["trace"], "i", "DP Ph1 GenericTwoTerminalVTypeSSN PF startup")

## GenericTwoTerminalITypeSSN

In [ ]:
result = run_generic_i_ssn("Ph1")
display(pd.DataFrame([result]))
plot_trace(result["trace"], "i", "DP Ph1 GenericTwoTerminalITypeSSN PF startup")

## VoltageSource

In [ ]:
result = run_pf_mapped_voltage_source("Ph1", "VoltageSource")
display(pd.DataFrame([result]))
# The source case returns the same PF-derived voltage trace used for its assertion.

## VoltageSourceNorton

In [ ]:
result = run_pf_mapped_voltage_source("Ph1", "VoltageSourceNorton")
display(pd.DataFrame([result]))

## AvVoltageSourceInverterDQ

In [ ]:
# DP Ph1 AvVoltageSourceInverterDQ mirrors the two-bus VSI power-flow example.
# Controller integrator values are deliberately omitted: initializeFromPowerflow
# must create a steady controller state from the operating voltage and P/Q refs.
INV1_P = 10e3
INV1_Q = 0.0
INV1_V = 400.0
inv1_pf, pf_inv1_nodes = make_pf(-INV1_P, -INV1_Q, name="Load")
in1 = dps.dp.SimNode("bus0")
in2 = dps.dp.SimNode("bus1")
in3 = dps.dp.SimNode("bus2")
in1_slack = dps.dp.ph1.NetworkInjection("Slack")
in1_l01 = dps.dp.ph1.PiLine("Line01")
in1_l01.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
in1_l12 = dps.dp.ph1.PiLine("Line12")
in1_l12.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
in1_inv = dps.dp.ph1.AvVoltageSourceInverterDQ("VSI")
in1_inv.set_parameters(W, INV1_V, INV1_P, INV1_Q)
in1_inv.set_filter_parameters(2e-3, 789.3e-6, 0.1, 0.1)
in1_inv.set_controller_parameters(0.25, 0.002, 0.001, 0.00008, 0.3, 0.01, W)
in1_slack.connect([in1])
in1_l01.connect([in1, in2])
in1_l12.connect([in2, in3])
in1_inv.connect([in3])
in1_sys = dps.SystemTopology(F, [in1, in2, in3], [in1_slack, in1_l01, in1_l12, in1_inv])
in1_sys.init_with_powerflow(inv1_pf, dps.Domain.DP)
in1_trace = run_sim(
    in1_sys,
    dps.Domain.DP,
    "DP_Ph1_AvVoltageSourceInverterDQ_AutoPF",
    [
        ("v", "v_intf", in1_inv),
        ("i", "i_intf", in1_inv),
        ("states", "powerctrl_states", in1_inv),
        ("inputs", "powerctrl_inputs", in1_inv),
        ("outputs", "powerctrl_outputs", in1_inv),
        ("pll", "pll_output", in1_inv),
    ],
    final_time=0.002,
    require_finite=False,
)
in1_numeric = in1_trace.select_dtypes(include=[np.number]).to_numpy()
in1_finite = bool(np.isfinite(in1_numeric).all())
in1_v = col(in1_trace, "v", True)
in1_i = col(in1_trace, "i", True)
in1_vpf = pf_inv1_nodes[2].single_voltage()
in1_v_error = float(abs(in1_v[0] - in1_vpf) / max(abs(in1_vpf), 1e-12))
in1_p = -np.real(in1_v * np.conj(in1_i))
in1_power_drift = float(np.nanmax(abs(in1_p - INV1_P)) / max(abs(INV1_P), 1e-12) * 100)
check(
    (np.isfinite(in1_v[0]) and in1_v_error < 1e-6),
    "validation at line 27",
    ("DP Ph1 AvVoltageSourceInverterDQ PF voltage", in1_v_error),
)
check(
    (np.isfinite(in1_i[0])),
    "validation at line 28",
    "DP Ph1 AvVoltageSourceInverterDQ initial current is not finite.",
)
in1_result = {
    "Domain/class": "DP.Ph1.AvVoltageSourceInverterDQ",
    "PF voltage error [%]": 100 * in1_v_error,
    "finite trace": in1_finite,
    "2 ms power deviation [%]": in1_power_drift,
    "result": "BUMPLESS" if in1_finite and in1_power_drift < 2 else "CHECK",
}
display(pd.DataFrame([in1_result]))
fig, axs = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
axs[0].plot(1e3 * in1_trace.time, in1_p, label="inverter P")
axs[0].axhline(INV1_P, ls="--", color="k")
axs[0].set_ylabel("P [W]")
axs[0].legend()
axs[0].grid(alpha=0.3)
for k in range(6):
    axs[1].plot(1e3 * in1_trace.time, in1_trace[f"states_{k}"], label=f"x{k}")
axs[1].set_ylabel("controller states")
axs[1].legend(ncol=3)
axs[1].grid(alpha=0.3)
axs[2].plot(1e3 * in1_trace.time, col(in1_trace, "pll"), label="PLL output")
axs[2].set(xlabel="Time [ms]", ylabel="PLL [rad/s]")
axs[2].legend()
axs[2].grid(alpha=0.3)
fig.suptitle("DP Ph1 averaged VSI PF initialization without manual controller states")
fig.tight_layout()
plt.show()

## AvVoltSourceInverterStateSpace

In [ ]:
# The same multi-node load grid validates the single-phase averaged state-space VSI.
ss1_pf, ss1_pf_nodes = make_pf(-10e3, 0.0, name="Load")
ss1_n0 = dps.dp.SimNode("bus0")
ss1_n1 = dps.dp.SimNode("bus1")
ss1_n2 = dps.dp.SimNode("bus2")
ss1_src = dps.dp.ph1.NetworkInjection("SS1_Slack")
ss1_l01 = dps.dp.ph1.PiLine("SS1_Line01")
ss1_l01.set_parameters(0.01, 0.1e-3, 1e-6, 1e-6)
ss1_l12 = dps.dp.ph1.PiLine("SS1_Line12")
ss1_l12.set_parameters(0.02, 0.2e-3, 1e-6, 1e-6)
ss1_inv = dps.dp.ph1.AvVoltSourceInverterStateSpace("SS1_VSI")
ss1_inv.set_parameters(
    2e-3, 789.3e-6, 0.1, 0.1, W, 0.25, 0.002, W, 10e3, 0.0, 0.001, 0.00008, 0.3, 0.01
)
ss1_src.connect([ss1_n0])
ss1_l01.connect([ss1_n0, ss1_n1])
ss1_l12.connect([ss1_n1, ss1_n2])
ss1_inv.connect([dps.dp.SimNode.gnd, ss1_n2])
ss1_sys = dps.SystemTopology(
    F, [ss1_n0, ss1_n1, ss1_n2], [ss1_src, ss1_l01, ss1_l12, ss1_inv]
)
ss1_sys.init_with_powerflow(ss1_pf, dps.Domain.DP)
ss1_trace = run_sim(
    ss1_sys,
    dps.Domain.DP,
    "DP_Ph1_AvVoltSourceInverterStateSpace_AutoPF",
    [
        ("v", "v", ss1_n2),
        ("i", "i_intf", ss1_inv),
        ("p", "p_inst", ss1_inv),
        ("q", "q_inst", ss1_inv),
        ("pll", "omega_pll", ss1_inv),
    ],
    final_time=2e-3,
    require_finite=False,
)
ss1_numeric = ss1_trace.select_dtypes(include=[np.number]).to_numpy()
ss1_finite = bool(np.isfinite(ss1_numeric).all())
ss1_v = col(ss1_trace, "v", True)
ss1_expected = ss1_pf_nodes[2].single_voltage()
ss1_v_error = float(abs(ss1_v[0] - ss1_expected) / max(abs(ss1_expected), 1e-12))
ss1_power = col(ss1_trace, "p")
ss1_power_drift = float(np.nanmax(abs(ss1_power - 10e3)) / 1e4 * 100)
check(
    (np.isfinite(ss1_v[0]) and ss1_v_error < 1e-6),
    "validation at line 21",
    ("DP Ph1 state-space VSI PF voltage", ss1_v_error),
)
check(
    (np.isfinite(col(ss1_trace, "i", True)[0])),
    "validation at line 22",
    "DP Ph1 state-space VSI initial current is non-finite.",
)
print(
    pd.DataFrame(
        [
            {
                "Domain/class": "DP.Ph1.AvVoltSourceInverterStateSpace",
                "PF voltage error [%]": 100 * ss1_v_error,
                "finite trace": ss1_finite,
                "2 ms P deviation [%]": ss1_power_drift,
                "result": "BUMPLESS" if ss1_finite and ss1_power_drift < 2 else "CHECK",
            }
        ]
    )
)
fig, axs = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axs[0].plot(1e3 * ss1_trace.time, ss1_power, label="active power")
axs[0].axhline(10e3, ls="--", color="k")
axs[0].set_ylabel("P [W]")
axs[0].grid(alpha=0.3)
axs[0].legend()
axs[1].plot(1e3 * ss1_trace.time, col(ss1_trace, "pll"), label="PLL output")
axs[1].set(xlabel="Time [ms]", ylabel="PLL [rad/s]")
axs[1].grid(alpha=0.3)
axs[1].legend()
fig.suptitle("DP Ph1 state-space VSI automatic PF initialization")
fig.tight_layout()
plt.show()

## SynchronGeneratorIdeal

In [ ]:
# Remaining DP Ph1 composite models: ideal synchronous source and state-space VSI.
# Both are supplied only with physical parameters; their operating point must come
# from the SP power flow through init_with_powerflow.
ideal_n0 = dps.dp.SimNode("SG_Gen")
ideal_n1 = dps.dp.SimNode("SG_Slack")
ideal_gen = dps.dp.ph1.SynchronGeneratorIdeal("SG_Gen")
ideal_line = dps.dp.ph1.PiLine("SG_Ideal_Line")
ideal_line.set_parameters(0.1, 1e-3, 1e-12, 1e-6)
ideal_slack = dps.dp.ph1.NetworkInjection("SG_Slack")
ideal_slack.set_parameters(SG1_VN)
ideal_gen.connect([ideal_n0])
ideal_line.connect([ideal_n0, ideal_n1])
ideal_slack.connect([ideal_n1])
ideal_sys = dps.SystemTopology(
    SG1_FREQ, [ideal_n0, ideal_n1], [ideal_gen, ideal_line, ideal_slack]
)
ideal_sys.init_with_powerflow(sg_pf_sys, dps.Domain.DP)
ideal_trace = run_sim(
    ideal_sys,
    dps.Domain.DP,
    "DP_Ph1_SynchronGeneratorIdeal_AutoPF",
    [("v", "v", ideal_n0), ("i", "i_intf", ideal_gen)],
    final_time=0.2e-3,
)
ideal_v = col(ideal_trace, "v", True)
ideal_expected = sg0.single_voltage()
ideal_err = float(abs(ideal_v[0] - ideal_expected) / max(abs(ideal_expected), 1e-12))
check(
    (np.isfinite(ideal_trace.select_dtypes(include=[np.number]).to_numpy()).all()),
    "validation at line 16",
    "validation condition is false",
)
check(
    (ideal_err < 1e-6),
    "validation at line 17",
    ("DP Ph1 SynchronGeneratorIdeal PF voltage", ideal_v[0], ideal_expected, ideal_err),
)
print(
    f"PASS DP Ph1 SynchronGeneratorIdeal automatic PF voltage initialization: {100*ideal_err:.3g}% error"
)

## SynchronGenerator3OrderVBR

In [ ]:
result = run_reduced_generator_case("SynchronGenerator3OrderVBR")
check(
    not str(result.get("result", "")).startswith("FAIL"),
    "SynchronGenerator3OrderVBR PF-start stability",
    result.get("result"),
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 SynchronGenerator3OrderVBR PF startup")

## SynchronGenerator4OrderVBR

In [ ]:
result = run_reduced_generator_case("SynchronGenerator4OrderVBR")
check(
    not str(result.get("result", "")).startswith("FAIL"),
    "SynchronGenerator4OrderVBR PF-start stability",
    result.get("result"),
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 SynchronGenerator4OrderVBR PF startup")

## SynchronGenerator4OrderTPM

In [ ]:
result = run_reduced_generator_case("SynchronGenerator4OrderTPM")
check(
    not str(result.get("result", "")).startswith("FAIL"),
    "SynchronGenerator4OrderTPM PF-start stability",
    result.get("result"),
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 SynchronGenerator4OrderTPM PF startup")

## SynchronGenerator4OrderPCM

In [ ]:
result = run_reduced_generator_case("SynchronGenerator4OrderPCM")
check(
    not str(result.get("result", "")).startswith("FAIL"),
    "SynchronGenerator4OrderPCM PF-start stability",
    result.get("result"),
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 SynchronGenerator4OrderPCM PF startup")

## SynchronGenerator5OrderVBR

In [ ]:
result = run_reduced_generator_case("SynchronGenerator5OrderVBR")
check(
    not str(result.get("result", "")).startswith("FAIL"),
    "SynchronGenerator5OrderVBR PF-start stability",
    result.get("result"),
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 SynchronGenerator5OrderVBR PF startup")

## SynchronGenerator6aOrderVBR

In [ ]:
result = run_reduced_generator_case("SynchronGenerator6aOrderVBR")
check(
    not str(result.get("result", "")).startswith("FAIL"),
    "SynchronGenerator6aOrderVBR PF-start stability",
    result.get("result"),
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 SynchronGenerator6aOrderVBR PF startup")

## SynchronGenerator6bOrderVBR

In [ ]:
result = run_reduced_generator_case("SynchronGenerator6bOrderVBR")
check(
    not str(result.get("result", "")).startswith("FAIL"),
    "SynchronGenerator6bOrderVBR PF-start stability",
    result.get("result"),
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 SynchronGenerator6bOrderVBR PF startup")

## SynchronGenerator6OrderPCM

In [ ]:
result = run_reduced_generator_case("SynchronGenerator6OrderPCM")
check(
    not str(result.get("result", "")).startswith("FAIL"),
    "SynchronGenerator6OrderPCM PF-start stability",
    result.get("result"),
)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "trace"}]))
plot_trace(result["trace"], "i", "DP Ph1 SynchronGenerator6OrderPCM PF startup")

## Coverage status

In [ ]:
coverage = pd.DataFrame(VALIDATION_RESULTS)
display(coverage)
print(
    f"Checks recorded: {(coverage.Status=='PASS').sum()} passed; {(coverage.Status=='FAIL').sum()} failed. Notebook execution continues regardless of failed checks."
)